In [1]:
import pandas as pd
import json
import time
from openai import OpenAI
from dotenv import load_dotenv
from tqdm import tqdm
import os

# =========================
# Load Environment Variables
# =========================

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

MODEL = "gpt-4.1-mini"

# =========================
# Load Dataset
# =========================

df = pd.read_csv("/Users/syedmubeen/projects/job_skills_needed/job_title_des.csv")

# Remove rows with missing values
df = df.dropna(subset=["Job Title", "Job Description"])

# Optional: limit dataset size
# df = df.head(50)

all_questions = []
seen_questions = set()


# =========================
# Prompt Builder
# =========================

def build_prompt(job_title, description):

    return f"""
You are generating evaluation questions for a Retrieval-Augmented Generation (RAG) system.

Generate EXACTLY 3 high-quality evaluation questions from this job posting.

Return ONLY a valid JSON array.

Each object MUST follow this exact structure:

[
  {{
    "question": "What skills are required for the Data Analyst role?",
    "keywords": ["Python", "SQL"],
    "reference_answer": "The Data Analyst role requires Python and SQL skills.",
    "category": "skills"
  }}
]

Allowed categories:
- direct_fact
- skills
- experience
- technology
- responsibility
- education

Guidelines:
- Questions must be answerable directly from the job description
- Keep questions concise and realistic
- Include important keywords
- Reference answers must be factual and grounded
- Avoid duplicate questions

Job Title:
{job_title}

Job Description:
{description}
"""


# =========================
# JSON Cleaner
# =========================

def clean_json_response(content):
    """
    Removes markdown formatting if present.
    """

    content = content.strip()

    # Remove markdown code fences
    if content.startswith("```json"):
        content = content.replace("```json", "").replace("```", "").strip()

    elif content.startswith("```"):
        content = content.replace("```", "").strip()

    return content


# =========================
# Generate Questions
# =========================

for _, row in tqdm(df.iterrows(), total=len(df)):

    job_title = str(row["Job Title"])
    description = str(row["Job Description"])

    try:

        response = client.chat.completions.create(
            model=MODEL,
            temperature=0.3,
            messages=[
                {
                    "role": "user",
                    "content": build_prompt(job_title, description),
                }
            ],
        )

        content = response.choices[0].message.content

        content = clean_json_response(content)

        questions = json.loads(content)

        # Validate and deduplicate
        for q in questions:

            required_keys = {
                "question",
                "keywords",
                "reference_answer",
                "category",
            }

            if not required_keys.issubset(q.keys()):
                continue

            question_text = q["question"].strip()

            if question_text not in seen_questions:

                seen_questions.add(question_text)

                all_questions.append(
                    {
                        "question": question_text,
                        "keywords": q["keywords"],
                        "reference_answer": q["reference_answer"],
                        "category": q["category"],
                    }
                )

        # Stop after 150 questions
        if len(all_questions) >= 150:
            break

        # Prevent rate limits
        time.sleep(1)

    except json.JSONDecodeError:
        print(f"JSON parsing failed for: {job_title}")

    except Exception as e:
        print(f"Error processing {job_title}: {e}")


# =========================
# Trim to 150
# =========================

all_questions = all_questions[:150]


# =========================
# Save JSONL
# =========================

output_file = "rag_eval_questions.jsonl"

with open(output_file, "w", encoding="utf-8") as f:

    for item in all_questions:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")


print(f"\nSaved {len(all_questions)} questions to {output_file}")

  2%|▏         | 49/2277 [03:17<2:29:18,  4.02s/it]


Saved 150 questions to rag_eval_questions.jsonl
